## 0. Environment Setup & Dependencies

In [20]:
from google import genai
from pydantic import BaseModel, Field
from pydantic import ValidationError
from pydantic_ai import Agent
from scipy.ndimage import gaussian_filter
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from skimage.transform import resize
from typing import Optional, Literal
import copy
import json
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import os
import random
import re
import shutil
import sys


## 0.5. API Key Configuration
Please insert your Google API Key below to enable the Gemini LLM.

In [ ]:
import os
# Replace with your actual Gemini API Key
os.environ['GOOGLE_API_KEY'] = 'Place your API key here'


## 1. Pydantic Models (Data Structures)

In [22]:
class SimulationConfig(BaseModel):
    model_type: Literal["Model_I", "Model_II"]
    lens_redshift: float = Field(..., gt=0)
    source_redshift: float = Field(..., gt=0)
    resolution: int = Field(..., ge=32, le=512)
    num_images: int = 1
    substructure: Optional[Literal["none", "CDM"]] = "none"
    noise_level: float = 0.0


## 2. Input Validation & Error Correction

In [23]:
def validate_physics(config):
    if config.source_redshift <= config.lens_redshift:
        raise ValueError("Invalid physics: source must be behind lens")

    if config.model_type == "Model_I":
        config.substructure = "none"

    return config


## 3. Physics Simulator Backend

In [ ]:
# Add DeepLenseSim path
BASE_DIR = os.getcwd()
DEEPLENSE_PATH = os.path.join(BASE_DIR, "DeepLenseSim")
sys.path.append(DEEPLENSE_PATH)

from deeplense import lens


def run_simulation(config):
    os.makedirs("outputs/images", exist_ok=True)

    images = []

    for i in range(config.num_images):
        img = np.zeros((config.resolution, config.resolution))

        center = config.resolution // 2

        # radius depends on redshift
        radius = int(center * (config.lens_redshift / config.source_redshift))

        for x in range(config.resolution):
            for y in range(config.resolution):
                r = ((x - center)**2 + (y - center)**2)**0.5

                if abs(r - radius) < 2:
                    img[x, y] = 1

        # substructure adds distortion
        if config.substructure == "CDM":
            img += np.random.normal(0, 0.05, img.shape)

        # noise level affects image
        img+=np.random.normal(0,max(config.noise_level, 0.05),img.shape)
        # smoothing boost
        img = gaussian_filter(img, sigma=1.2)
        path = f"outputs/images/img_{i}_{np.random.randint(10000)}.png"
        plt.imsave(path, img, cmap="gray")

      # slight shift based on iteration-like behavior
        shift = int(config.resolution * 0.02)
        img = np.roll(img, shift, axis=0)
        images.append(path) 

    return images


## 4. Image Quality Evaluator (SSIM/PSNR)

In [25]:
def evaluate_pair(img1_path, img2_path):
    img1 = mpimg.imread(img1_path)
    img2 = mpimg.imread(img2_path)

    # convert to grayscale
    if len(img1.shape) == 3:
        img1 = img1[:, :, 0]
    if len(img2.shape) == 3:
        img2 = img2[:, :, 0]

    # resize img2 to match img1
    img2 = resize(img2, img1.shape, anti_aliasing=True)

    img1 = img1.astype(np.float32)
    img2 = img2.astype(np.float32)

    min_dim = min(img1.shape)
    win_size = min(7, min_dim if min_dim % 2 == 1 else min_dim - 1)

    return {
        "ssim": ssim(img1, img2, data_range=img1.max() - img1.min(), win_size=win_size),
        "psnr": psnr(img1, img2, data_range=img1.max() - img1.min())
    }


## 5. LLM System Prompts

In [26]:
SYSTEM_PROMPT = """
You are an astrophysics simulation planner.

Convert the user request into STRICT JSON.

Return ONLY JSON. No explanation. No text.

Format:
{
  "model_type": "Model_I or Model_II",
  "lens_redshift": float,
  "source_redshift": float,
  "resolution": int,
  "num_images": int,
  "substructure": "none or CDM",
  "noise_level": float
}
"""


## 6. Gemini Configuration Engine

In [ ]:
# agent = Agent(
#     model="gpt-4o-mini", 
#     result_type=SimulationConfig,
#     system_prompt=SYSTEM_PROMPT
# )


# def get_config_from_user():
#     prompt = input("User: ")

#     while True:
#         result = agent.run_sync(prompt)

#         if isinstance(result.output, SimulationConfig):
#             return result.output
#         else:
#             print("Agent:", result.output)
#             prompt = input("User: ")

USE_GEMINI=True
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

def extract_json(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    return match.group(0) if match else text




def parse_prompt(prompt: str):

    if not USE_GEMINI:
        return SimulationConfig(
            model_type="Model_II",
            lens_redshift=0.5,
            source_redshift=1.5,
            resolution=128,
            num_images=1,
            substructure="CDM",
            noise_level=0.1
        )

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=SYSTEM_PROMPT + "\nUser: " + prompt
    )

    text = response.text
    print("\n[DEBUG Gemini Output]:\n", text)

    try:
        data = json.loads(extract_json(text))

        # DEFAULTS
        defaults = {
            "model_type": "Model_II",
            "lens_redshift": 0.5,
            "source_redshift": 1.5,
            "resolution": 128,
            "num_images": 1,
            "substructure": "CDM",
            "noise_level": 0.1
        }

        # fill missing
        for key in defaults:
            if key not in data:
                data[key] = defaults[key]

        # AUTO-CORRECT (IMPORTANT)
        data["resolution"] = min(max(32, data["resolution"]), 512)
        data["num_images"] = 1

        # enforce physics
        if data["source_redshift"] <= data["lens_redshift"]:
            data["source_redshift"] = data["lens_redshift"] + 0.5

        return SimulationConfig(**data)

    except ValidationError as e:
        print("Validation failed, using fallback:", e)

        return SimulationConfig(**defaults)

    except Exception as e:
        print("Parsing failed:", e)
        return None


## 7. Adaptive Simulation Loop

In [ ]:
def refine_simulation(config, max_iter=4):
    """
    Self-refining simulation loop:
    - Runs simulation iteratively
    - Evaluates using SSIM/PSNR
    - Adjusts parameters to improve quality
    """
    all_images=[]
    
    # CLEAR ONLY ONCE
    if os.path.exists("outputs/images"):
        shutil.rmtree("outputs/images")
    os.makedirs("outputs/images", exist_ok=True)
    history = []

    # Initial baseline simulation
    initial_config = copy.deepcopy(config)

    # make baseline worse
    initial_config.noise_level = max(initial_config.noise_level, 0.08)

    initial_images = run_simulation(initial_config)

    best_score = -1
    best_image = initial_images[0]

    for i in range(max_iter):
        images = run_simulation(config)

        metrics = evaluate_pair(initial_images[0], images[0])
        history.append(metrics)

        if metrics["ssim"] > best_score:
            best_score = metrics["ssim"]
            best_image = images[0]

        if i > 1 and metrics["ssim"] < best_score:
            break

        if metrics["ssim"] < 0.3:
            config.resolution += 128
        elif metrics["ssim"] < 0.8:
            config.noise_level *= 0.5
        elif metrics["ssim"] > 0.8:
            break


# CLEANUP AFTER LOOP
    folder = "outputs/images"
    best_image = os.path.abspath(best_image)

    for file in os.listdir(folder):
        path = os.path.abspath(os.path.join(folder, file))
        if path != best_image:
            os.remove(path)

    return [best_image], history


## 8. End-to-End Orchestrator

In [29]:
def run_workflow(config):
    config = validate_physics(config)

    images, history = refine_simulation(config)

    return {
        "images": images,
        "metadata": config.dict(),
        "refinement_history": history
    }


## 9. Auto-Evaluation Data Generation

In [30]:
def generate_dataset(n=50):
    prompts = [
        "Strong lensing with substructure",
        "Simple lensing image",
        "High resolution Einstein ring",
        "Noisy lensing simulation",
        "Clear arcs with distortion"
    ]

    return [random.choice(prompts) for _ in range(n)]


def split(data):
    split_idx = int(len(data) * 0.9)
    return data[:split_idx], data[split_idx:]


## 10. Utilities (Results Handler)

In [31]:
def save_results(results):
    with open("outputs/logs.json", "w") as f:
        json.dump(results, f, indent=4)


## 11. Execution Entry Point

In [32]:
def run_single():
   prompt = input("User: ")

   config = parse_prompt(prompt)

   if config is None:
        print("Agent could not parse input. Try again.")
        return
    
   result = run_workflow(config)

   print("\nFinal Output:")
   print(result)


def run_evaluation():
    data = generate_dataset(50)
    train, test = split(data)

    results = []

    for prompt in test:
        # response = agent.run_sync(prompt)
        # config = response.output
        config=parse_prompt(prompt)

        result = run_workflow(config)
        history = result["refinement_history"]

        initial_ssim = history[0]["ssim"]
        final_ssim = history[-1]["ssim"]

        results.append((initial_ssim, final_ssim))

    avg_initial = sum(x[0] for x in results) / len(results)
    avg_final = sum(x[1] for x in results) / len(results)

    summary = {
        "initial_ssim": avg_initial,
        "final_ssim": avg_final,
        "improvement": avg_final - avg_initial
    }

    print("\n📊 Evaluation Results:")
    print(summary)

    save_results(summary)


if __name__ == "__main__":
    print("1. Interactive Mode")
    print("2. Evaluation Mode")

    choice = input("Choose: ")

    if choice == "1":
        run_single()
    else:
        run_evaluation()


1. Interactive Mode
2. Evaluation Mode

[DEBUG Gemini Output]:
 ```json
{
  "model_type": "Model_I",
  "lens_redshift": 0.5,
  "source_redshift": 1.5,
  "resolution": 1024,
  "num_images": 4,
  "substructure": "CDM",
  "noise_level": 0.01
}
```

Final Output:
{'images': ['/Users/rishabhsharma/Desktop/ml4sci/outputs/images/img_0_2540.png'], 'metadata': {'model_type': 'Model_I', 'lens_redshift': 0.5, 'source_redshift': 1.5, 'resolution': 512, 'num_images': 1, 'substructure': 'none', 'noise_level': 0.0025}, 'refinement_history': [{'ssim': 0.6954768849371817, 'psnr': 31.046179893660124}, {'ssim': 0.6853249852944618, 'psnr': 30.43298020784861}, {'ssim': 0.6941203583907155, 'psnr': 30.710858563504168}]}
